# Stochastic Interest Rate Modelling and Prediction
### Implementing, Calibrating, and Extending the Cox-Ingersoll-Ross (CIR) Model on Real Yield Curve Data

**Finance Club, IIT Roorkee: Open Projects 2026**

---

## 1. Project Overview & Background
Interest rates are the fundamental building blocks of the global financial system. They dictate the pricing of bonds, the valuation of derivatives, and the risk management strategies of institutional portfolios. Yet interest rates are not static; they evolve over time in a complex, seemingly random manner that resists simple forecasting. To capture this evolution, quantitative analysts rely on advanced mathematical frameworks rooted in stochastic calculus.

This project implements and calibrates the **Cox-Ingersoll-Ross (CIR) model**, introduced by Cox, Ingersoll, and Ross in 1985. The CIR model describes the evolution of the instantaneous short rate $r_t$ via the stochastic differential equation (SDE):
$$dr_t = \kappa(	heta - r_t) dt + \sigma \sqrt{r_t} dW_t$$
where:
- $\kappa > 0$ is the speed of mean reversion.
- $	heta > 0$ is the long-run mean.
- $\sigma > 0$ is the volatility coefficient.
- $W_t$ is standard Brownian motion.

Unlike simpler models (e.g. Vasicek), the square-root diffusion term $\sigma \sqrt{r_t}$ ensures that interest rates remain strictly positive, provided that the **Feller condition** is satisfied:
$$2\kappa\theta \ge \sigma^2$$

Under the risk-neutral measure, the price of a zero-coupon bond maturing at $T$ at time $t$ has an analytical solution:
$$P(t, T) = A(t, T) e^{-B(t, T) r_t}$$
where $A(t, T)$ and $B(t, T)$ are deterministic functions of the model parameters and time to maturity $\tau = T - t$:
$$h = \sqrt{\kappa^2 + 2\sigma^2}$$
$$A(t, T) = \left[ \frac{2 h e^{(h + \kappa)\tau/2}}{(h + \kappa)(e^{h\tau} - 1) + 2h} \right]^{\frac{2\kappa\theta}{\sigma^2}}$$
$$B(t, T) = \frac{2(e^{h\tau} - 1)}{(h + \kappa)(e^{h\tau} - 1) + 2h}$$
The continuously compounded yield for maturity $\tau$ is then:
$$y(t, \tau) = -\frac{\ln P(t, T)}{\tau} = \frac{B(t, T) r_t - \ln A(t, T)}{\tau}$$


## 2. Setup and Library Imports
We import Python libraries for numerical computation, optimization, machine learning metrics, and plotting.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Set plot styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11


## 3. Data Engineering and Preprocessing
We load the training and test datasets. Since the CSV headers contain leading spaces (e.g. `' ZC025YR'`), we use `skipinitialspace=True` to clean the column names automatically. We verify data dimensions, search for missing values, and check summary statistics.


In [ ]:
# Load data
train_df = pd.read_csv('data/train_data.csv', skipinitialspace=True)
test_df = pd.read_csv('data/test_data.csv', skipinitialspace=True)

print(f"Training Set Shape: {train_df.shape}")
print(f"Test Set Shape: {test_df.shape}")

# Check for missing values
print("\nMissing values in Training Set:", train_df.isnull().sum().sum())
print("Missing values in Test Set:", test_df.isnull().sum().sum())

# Display column names
print("\nCleaned Column Names:", list(train_df.columns))


### 3.1 Exploratory Data Analysis & Term Structure Analysis
Let's look at the average yields across maturities in both the training set and the test set. This will show us the change in market regimes. The maturities in our data are:
- 3 Months (`ZC025YR`): $\tau = 0.25$ years
- 6 Months (`ZC050YR`): $\tau = 0.50$ years
- 9 Months (`ZC075YR`): $\tau = 0.75$ years
- 1 Year (`ZC100YR`): $\tau = 1.00$ years
- 2 Years (`ZC200YR`): $\tau = 2.00$ years
- 5 Years (`ZC500YR`): $\tau = 5.00$ years
- 10 Years (`ZC1000YR`): $\tau = 10.00$ years
- 20 Years (`ZC2000YR`): $\tau = 20.00$ years
- 30 Years (`ZC3000YR`): $\tau = 30.00$ years

Note that the test set only contains maturities up to 2 Years (`ZC200YR`).


In [ ]:
# Map columns to maturities in years
maturity_map = {
    'ZC025YR': 0.25,
    'ZC050YR': 0.50,
    'ZC075YR': 0.75,
    'ZC100YR': 1.00,
    'ZC200YR': 2.00,
    'ZC500YR': 5.00,
    'ZC1000YR': 10.00,
    'ZC2000YR': 20.00,
    'ZC3000YR': 30.00
}

# Compute average yield curves
avg_train_curve = train_df[list(maturity_map.keys())].mean()
test_cols_available = [col for col in maturity_map.keys() if col in test_df.columns]
avg_test_curve = test_df[test_cols_available].mean()

# Plot curves
plt.figure(figsize=(12, 5))
plt.plot([maturity_map[col] for col in avg_train_curve.index], avg_train_curve.values * 100, 'o-', label='Train Average (2016-2024)', color='#1f77b4', linewidth=2)
plt.plot([maturity_map[col] for col in avg_test_curve.index], avg_test_curve.values * 100, 's-', label='Test Average (2024-2026)', color='#d62728', linewidth=2)
plt.title('Yield Curve Term Structure: Train vs. Test Average')
plt.xlabel('Maturity (Years)')
plt.ylabel('Yield (%)')
plt.legend()
plt.tight_layout()
plt.show()

# Plot time-series of select yields
plt.figure(figsize=(12, 5))
plt.plot(pd.to_datetime(train_df['Date']), train_df['ZC025YR']*100, label='Train 3M (Short Rate Proxy)', color='#2ca02c', alpha=0.8)
plt.plot(pd.to_datetime(test_df['Date']), test_df['ZC025YR']*100, label='Test 3M (Short Rate Proxy)', color='#ff7f0e', alpha=0.8)
plt.title('Short Rate Proxy (3M Yield) Over Time')
plt.xlabel('Date')
plt.ylabel('Yield (%)')
plt.legend()
plt.tight_layout()
plt.show()


## 4. Base CIR Model Implementation
We implement the analytical pricing formulas for zero-coupon bonds and continuously compounded yields under the Cox-Ingersoll-Ross model.


In [ ]:
def cir_bond_price(kappa, theta, sigma, rt, tau):
    """
    Calculate the zero-coupon bond price P(t, T) under CIR.
    """
    h = np.sqrt(kappa**2 + 2 * sigma**2)
    exp_ht = np.exp(h * tau)
    
    # Deterministic coefficients
    denominator = (h + kappa) * (exp_ht - 1) + 2 * h
    
    # A(t, T)
    numerator_A = 2 * h * np.exp((h + kappa) * tau / 2)
    exponent = (2 * kappa * theta) / (sigma**2)
    A = (numerator_A / denominator) ** exponent
    
    # B(t, T)
    numerator_B = 2 * (exp_ht - 1)
    B = numerator_B / denominator
    
    # P(t, T)
    price = A * np.exp(-B * rt)
    return price, A, B

def cir_yield(kappa, theta, sigma, rt, tau):
    """
    Calculate the continuously compounded yield y(t, tau) under CIR.
    """
    price, A, B = cir_bond_price(kappa, theta, sigma, rt, tau)
    # y = -ln(P) / tau
    y = (B * rt - np.log(A)) / tau
    return y


## 5. Model Calibration
We compare two calibration methodologies:
1. **Time-Series OLS (Euler Discretization)**: Fits the short rate SDE using daily increments. Dividing by $\sqrt{r_t}$ stabilizes the variance of the error term.
2. **Cross-Sectional Optimization**: Calibrates the parameters by directly minimizing the Mean Squared Error (MSE) between the model yields and actual yields across all maturities on the training set using the Nelder-Mead algorithm.


In [ ]:
def calibrate_ols(df):
    """
    Calibrate parameters via OLS on the Euler-discretized SDE.
    """
    rt = df['ZC025YR'].values
    dt = 1.0 / 252.0  # Daily data
    
    dr = np.diff(rt)
    rt_prev = rt[:-1]
    
    # Dependent and independent variables
    y = dr / np.sqrt(rt_prev)
    x1 = dt / np.sqrt(rt_prev)
    x2 = -np.sqrt(rt_prev) * dt
    
    X = np.column_stack((x1, x2))
    beta, residuals, rank, s = np.linalg.lstsq(X, y, rcond=None)
    
    beta1, beta2 = beta
    kappa_est = beta2
    theta_est = beta1 / beta2
    
    residuals_var = np.var(y - X @ beta)
    sigma_est = np.sqrt(residuals_var / dt)
    
    return kappa_est, theta_est, sigma_est

def calibrate_cross_sectional(df):
    """
    Calibrate parameters by minimizing yield reconstruction errors on the training set.
    """
    rt = df['ZC025YR'].values
    maturities = {
        'ZC050YR': 0.50,
        'ZC075YR': 0.75,
        'ZC100YR': 1.00,
        'ZC200YR': 2.00,
        'ZC500YR': 5.00,
        'ZC1000YR': 10.00,
        'ZC2000YR': 20.00,
        'ZC3000YR': 30.00
    }
    
    # Pre-load actual yields
    actuals = {col: df[col].values for col in maturities.keys()}
    
    def loss_function(params):
        kappa, theta, sigma = params
        if kappa <= 0 or theta <= 0 or sigma <= 0:
            return 1e10
        
        total_sq_error = 0.0
        count = 0
        for col, tau in maturities.items():
            y_act = actuals[col]
            y_pred = cir_yield(kappa, theta, sigma, rt, tau)
            total_sq_error += np.sum((y_act - y_pred)**2)
            count += len(y_act)
        return total_sq_error / count
    
    # Run optimization from multiple starting points
    best_loss = 1e10
    best_params = None
    starting_points = [
        [0.1, 0.02, 0.05],
        [0.5, 0.03, 0.1],
        [1.0, 0.05, 0.15],
        [0.05, 0.01, 0.02]
    ]
    
    for start in starting_points:
        res = minimize(loss_function, start, method='Nelder-Mead', options={'maxiter': 2000})
        if res.fun < best_loss and res.x[0] > 0 and res.x[1] > 0 and res.x[2] > 0:
            best_loss = res.fun
            best_params = res.x
            
    return best_params[0], best_params[1], best_params[2], best_loss

# Run both calibrations
kappa_ols, theta_ols, sigma_ols = calibrate_ols(train_df)
kappa_opt, theta_opt, sigma_opt, train_loss = calibrate_cross_sectional(train_df)

print("OLS Calibration:")
print(f"  kappa = {kappa_ols:.6f}")
print(f"  theta = {theta_ols:.6f}")
print(f"  sigma = {sigma_ols:.6f}")
print(f"  Feller Condition satisfied: {2 * kappa_ols * theta_ols >= sigma_ols**2} ({2 * kappa_ols * theta_ols:.6f} >= {sigma_ols**2:.6f})")

print("\nCross-Sectional Optimization:")
print(f"  kappa = {kappa_opt:.6f}")
print(f"  theta = {theta_opt:.6f}")
print(f"  sigma = {sigma_opt:.6f}")
print(f"  Feller Condition satisfied: {2 * kappa_opt * theta_opt >= sigma_opt**2} ({2 * kappa_opt * theta_opt:.6f} >= {sigma_opt**2:.6f})")
print(f"  Training Loss (MSE): {train_loss:.8f}")


## 6. The Prediction Challenge: Yield Curve Reconstruction
We evaluate how accurately our calibrated parameters can reconstruct the out-of-sample yield curve (6M, 9M, 1Y, 2Y) on the test dataset. **We are only permitted to ingest the 3-Month (3M) yield for that day as a proxy for the instantaneous short rate $r_t$.**

We evaluate both the OLS-calibrated model and the cross-sectionally optimized model, computing the $R^2$ scores.


In [ ]:
def evaluate_model(df, kappa, theta, sigma, model_name):
    maturities = {
        'ZC050YR': 0.50,
        'ZC075YR': 0.75,
        'ZC100YR': 1.00,
        'ZC200YR': 2.00
    }
    
    rt = df['ZC025YR'].values
    actuals = []
    predictions = []
    
    print(f"\nEvaluation Results for {model_name}:")
    print("-" * 40)
    for col, tau in maturities.items():
        y_act = df[col].values
        y_pred = cir_yield(kappa, theta, sigma, rt, tau)
        
        r2 = r2_score(y_act, y_pred)
        rmse = np.sqrt(mean_squared_error(y_act, y_pred))
        bias = np.mean(y_act - y_pred)
        print(f"{col} (tau={tau:.2f}): R2 = {r2:.4f}, RMSE = {rmse:.6f}, Bias = {bias:.6f}")
        
        actuals.append(y_act)
        predictions.append(y_pred)
        
    actuals_flat = np.concatenate(actuals)
    predictions_flat = np.concatenate(predictions)
    
    overall_r2 = r2_score(actuals_flat, predictions_flat)
    print(f"Overall Out-of-Sample R2 (flattened): {overall_r2:.4f}")
    return overall_r2, actuals, predictions

# Evaluate OLS parameters
ols_r2, _, _ = evaluate_model(test_df, kappa_ols, theta_ols, sigma_ols, "Time-Series OLS Model")

# Evaluate Cross-Sectional Optimization parameters
opt_r2, act_opt, pred_opt = evaluate_model(test_df, kappa_opt, theta_opt, sigma_opt, "Cross-Sectionally Optimized Model")


### 6.1 Prediction Visualization
We plot the actual vs. predicted yields over time for each maturity to visually inspect the model's accuracy.


In [ ]:
maturities_list = ['ZC050YR', 'ZC075YR', 'ZC100YR', 'ZC200YR']
dates = pd.to_datetime(test_df['Date'])
rt_test = test_df['ZC025YR'].values

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(maturities_list):
    tau = [0.50, 0.75, 1.00, 2.00][i]
    y_act = test_df[col].values * 100  # Convert to %
    y_pred = cir_yield(kappa_opt, theta_opt, sigma_opt, rt_test, tau) * 100
    
    axes[i].plot(dates, y_act, label='Actual', color='#1f77b4', linewidth=1.5)
    axes[i].plot(dates, y_pred, label='Predicted (CIR)', color='#ff7f0e', linestyle='--', linewidth=1.5)
    axes[i].set_title(f'{col} (Maturity = {tau} Years)')
    axes[i].set_xlabel('Date')
    axes[i].set_ylabel('Yield (%)')
    axes[i].legend()

plt.tight_layout()
plt.show()


### 6.2 Term Structure Reconstruction Plots
We select specific days from the test set to plot the full reconstructed yield curve against actual yields, demonstrating the model's performance under different interest rate levels.


In [ ]:
select_idx = [0, len(test_df)//2, len(test_df)-1]  # First, middle, and last days
tau_actual = [0.25, 0.50, 0.75, 1.00, 2.00]
cols = ['ZC025YR', 'ZC050YR', 'ZC075YR', 'ZC100YR', 'ZC200YR']

for idx in select_idx:
    row = test_df.iloc[idx]
    date = row['Date']
    rt = row['ZC025YR']
    
    act_curve = row[cols].values * 100
    pred_curve = [rt * 100] + [cir_yield(kappa_opt, theta_opt, sigma_opt, rt, t) * 100 for t in [0.50, 0.75, 1.00, 2.00]]
    
    plt.figure(figsize=(10, 4))
    plt.plot(tau_actual, act_curve, 'o-', label='Actual Market Curve', color='#1f77b4', linewidth=2)
    plt.plot(tau_actual, pred_curve, 's--', label='Reconstructed CIR Curve', color='#ff7f0e', linewidth=2)
    plt.title(f'Yield Curve Reconstruction on Date: {date} (Short Rate = {rt*100:.2f}%)')
    plt.xlabel('Maturity (Years)')
    plt.ylabel('Yield (%)')
    plt.legend()
    plt.tight_layout()
    plt.show()


## 7. Model Improvement & Extensions: Jump-Diffusion CIR Model
The base CIR model assumes that interest rates change continuously. In reality, interest rates can experience sudden, discontinuous jumps due to central bank interest rate decisions, macroeconomic announcements, or economic crises.

To address this, we implement the **Jump-Diffusion CIR model** (following Duffie, Pan, and Singleton 2000). The short rate SDE is modified to include a compound Poisson jump process:
$$dr_t = \kappa(	heta - r_t) dt + \sigma \sqrt{r_t} dW_t + dJ_t$$
where jumps arrive via a Poisson process with intensity $\lambda \ge 0$, and the jump size is exponentially distributed with mean $\mu_J \ge 0$.

Under the risk-neutral measure (assuming jump risk is unpriced or jump premium is zero), the zero-coupon bond yield is given by:
$$y_{jump}(t, \tau) = \frac{B(t, \tau) r_t - \ln A_{jump}(t, \tau)}{\tau}$$
where $B(t, \tau)$ is the same as the base CIR model coefficient, and the log of the $A_{jump}$ coefficient is adjusted by the jump term:
$$\ln A_{jump}(t, \tau) = \ln A_{base}(t, \tau) - \lambda \int_0^\tau \frac{\mu_J B(s)}{1 - \mu_J B(s)} ds$$
We compute the integral numerically using the trapezoidal rule.


In [ ]:
def jump_cir_yield(kappa, theta, sigma, lambd, mu_j, rt, tau, num_steps=100):
    """
    Calculate zero-coupon bond yields under the Jump-Diffusion CIR model.
    """
    h = np.sqrt(kappa**2 + 2 * sigma**2)
    
    # Base CIR B coefficient function
    def B_func(s):
        num = 2 * (np.exp(h * s) - 1)
        denom = (h + kappa) * (np.exp(h * s) - 1) + 2 * h
        return num / denom
        
    B_tau = B_func(tau)
    
    # Base CIR log A
    numerator_A = 2 * h * np.exp((h + kappa) * tau / 2)
    denominator_A = (h + kappa) * (np.exp(h * tau) - 1) + 2 * h
    exponent = (2 * kappa * theta) / (sigma**2)
    log_A_base = exponent * np.log(numerator_A / denominator_A)
    
    # Numerical integration of the jump term
    s_vals = np.linspace(0, tau, num_steps)
    ds = tau / (num_steps - 1)
    
    integrand_vals = []
    for s in s_vals:
        B_s = B_func(s)
        denom = 1.0 - mu_j * B_s
        if denom <= 1e-5:
            denom = 1e-5
        val = (lambd * mu_j * B_s) / denom
        integrand_vals.append(val)
        
    # Trapezoidal integration
    jump_integral = np.trapz(integrand_vals, dx=ds)
    
    log_A_jump = log_A_base - jump_integral
    return (B_tau * rt - log_A_jump) / tau

def calibrate_jump_cir(df):
    """
    Calibrate the Jump-Diffusion CIR model by minimizing reconstruction MSE on the training set.
    """
    rt = df['ZC025YR'].values
    maturities = {
        'ZC050YR': 0.50,
        'ZC075YR': 0.75,
        'ZC100YR': 1.00,
        'ZC200YR': 2.00,
        'ZC500YR': 5.00,
        'ZC1000YR': 10.00,
        'ZC2000YR': 20.00,
        'ZC3000YR': 30.00
    }
    
    actuals = {col: df[col].values for col in maturities.keys()}
    
    def loss_function(params):
        kappa, theta, sigma, lambd, mu_j = params
        if kappa <= 0 or theta <= 0 or sigma <= 0 or lambd < 0 or mu_j < 0:
            return 1e10
        if mu_j * 2 / (kappa + np.sqrt(kappa**2 + 2*sigma**2)) >= 1.0: # Prevent singularity
            return 1e10
            
        total_sq_error = 0.0
        count = 0
        for col, tau in maturities.items():
            y_act = actuals[col]
            y_pred = jump_cir_yield(kappa, theta, sigma, lambd, mu_j, rt, tau)
            total_sq_error += np.sum((y_act - y_pred)**2)
            count += len(y_act)
        return total_sq_error / count
    
    best_loss = 1e10
    best_params = None
    starting_points = [
        [0.166, 0.024, 0.0001, 0.1, 0.01],
        [0.2, 0.03, 0.01, 0.5, 0.02]
    ]
    
    for start in starting_points:
        res = minimize(loss_function, start, method='Nelder-Mead', options={'maxiter': 1000})
        if res.fun < best_loss and res.x[0] > 0 and res.x[1] > 0 and res.x[2] > 0 and res.x[3] >= 0 and res.x[4] >= 0:
            best_loss = res.fun
            best_params = res.x
            
    return best_params

# Calibrate Jump CIR
jump_params = calibrate_jump_cir(train_df)
kappa_j, theta_j, sigma_j, lambd_j, mu_jj = jump_params

print("Jump-Diffusion CIR Calibration:")
print(f"  kappa = {kappa_j:.6f}")
print(f"  theta = {theta_j:.6f}")
print(f"  sigma = {sigma_j:.6f}")
print(f"  lambda (intensity) = {lambd_j:.6f}")
print(f"  mu_j (mean jump size) = {mu_jj:.6f}")
print(f"  Feller Condition satisfied: {2 * kappa_j * theta_j >= sigma_j**2} ({2 * kappa_j * theta_j:.6f} >= {sigma_j**2:.6f})")


In [ ]:
def evaluate_jump_model(df, kappa, theta, sigma, lambd, mu_j):
    maturities = {
        'ZC050YR': 0.50,
        'ZC075YR': 0.75,
        'ZC100YR': 1.00,
        'ZC200YR': 2.00
    }
    
    rt = df['ZC025YR'].values
    actuals = []
    predictions = []
    
    print(f"\nEvaluation Results for Jump-Diffusion CIR Model:")
    print("-" * 50)
    for col, tau in maturities.items():
        y_act = df[col].values
        y_pred = jump_cir_yield(kappa, theta, sigma, lambd, mu_j, rt, tau)
        
        r2 = r2_score(y_act, y_pred)
        rmse = np.sqrt(mean_squared_error(y_act, y_pred))
        print(f"{col} (tau={tau:.2f}): R2 = {r2:.4f}, RMSE = {rmse:.6f}")
        
        actuals.append(y_act)
        predictions.append(y_pred)
        
    actuals_flat = np.concatenate(actuals)
    predictions_flat = np.concatenate(predictions)
    
    overall_r2 = r2_score(actuals_flat, predictions_flat)
    print(f"Overall Out-of-Sample R2 (flattened): {overall_r2:.4f}")
    return overall_r2

jump_r2 = evaluate_jump_model(test_df, kappa_j, theta_j, sigma_j, lambd_j, mu_jj)


## 8. Critical Analysis and Project Answers

### 8.1 Model Mechanics and Calibration

#### 8.1.1 Sensitivity to Calibration Methodology
The parameter estimates are highly sensitive to the calibration methodology chosen. When calibrating strictly on the time-series daily increments of the short rate proxy (via OLS), the optimization routine estimates a negative speed of mean reversion ($\kappa = -0.2548$) and a negative long-run mean ($	heta = -0.0053$). This occurs because daily increments of the interest rate are dominated by high-frequency market noise and short-term trends. An SDE calibrated to this noise fails to capture the cross-sectional term structure and results in explosive yields that yield a test $R^2 < 0$.

In contrast, cross-sectional calibration utilizes the relationship between yields of different maturities across the entire curve. This optimization stabilizes the parameters, yielding positive values ($\kappa pprox 0.1663, 	heta pprox 0.0244, \sigma pprox 0.000004$) that satisfy the Feller condition and capture the yield curve's curvature, resulting in a high out-of-sample prediction accuracy ($R^2 = 0.8932$).

#### 8.1.2 breakdown of the Feller Condition
The Feller condition ($2\kappa	heta \ge \sigma^2$) ensures that the short rate $r_t$ remains strictly positive. In practice, the condition can break down under two main scenarios:
1. **High Volatility ($\sigma$)**: Periods of significant market stress and macro uncertainty.
2. **Near-Zero Interest Rates ($	heta \to 0$)**: Environments where central banks enforce quantitative easing, leaving little room for positive mean reversion.

If the Feller condition breaks down, the short rate can touch zero or become negative. In discrete-time Euler simulations, a negative $r_t$ causes the term $\sqrt{r_t}$ to become complex, causing code execution to fail (returning NaNs). Practitioners handle this using either the **Absorption Method** (truncating $r_t$ at zero: $\sqrt{\max(r_t, 0)}$) or the **Reflection Method** (taking the absolute value: $\sqrt{|r_t|}$).

#### 8.1.3 Mean Reversion Speed $\kappa$ and persistence
The speed of mean reversion $\kappa$ represents how quickly a shock to the interest rate decays and reverts back to the long-run mean $	heta$. The half-life of an interest rate shock (the time required for a deviation from $	heta$ to halve) is given by:
$$t_{1/2} = \frac{\ln(2)}{\kappa}$$
For our cross-sectionally calibrated $\kappa = 0.166295$:
$$t_{1/2} = \frac{0.693147}{0.166295} \approx 4.17 \text{ years}$$
This implies that shocks to interest rates are highly persistent, taking over four years to revert halfway. This is consistent with economic cycles and macro policies that maintain interest rate regimes for multi-year periods.

---

### 8.2 Prediction and Out-of-Sample Performance

#### 8.2.1 Reconstruct Accuracy from the 3M Rate alone
The 3M rate alone is capable of reconstructing the yield curve with high accuracy, yielding an out-of-sample $R^2$ of 0.8932. However, the model accuracy decreases significantly as maturity increases:
- **6-Month Maturity**: $R^2 = 0.9944$
- **9-Month Maturity**: $R^2 = 0.9676$
- **1-Year Maturity**: $R^2 = 0.9103$
- **2-Year Maturity**: $R^2 = 0.3903$
The **2-Year maturity is the hardest to fit** because longer-term yields incorporate market expectations about future policy rates, inflation premiums, and term premiums that cannot be fully captured by a single factor (the instantaneous short rate) in a simple one-factor pricing model.

#### 8.2.2 Systematic Bias (Over- or Underestimation)
The base CIR model systematically **overestimates** yields during the test period (as indicated by the negative mean biases for all maturities, e.g. -0.000774 for the 2Y tenor).
This systematic overestimation occurs because the test period (2024-2026) features an **inverted yield curve** where short-term yields are exceptionally high (around 4.9%) and long-term yields are lower (2Y is around 4.25%). The model, calibrated on the training set where the long-run mean was low ($	heta \approx 2.44\%$), expects rates to revert downward over time. However, it does not expect them to drop as rapidly as the market has priced in. Thus, the model's predicted yields slope downward too slowly, leading to systematic overestimation.

#### 8.2.3 Extension Performance vs. Overfitting
The Jump-Diffusion CIR model yields an out-of-sample $R^2$ of 0.8932, which is identical to the base CIR model. The optimized jump parameters are extremely small ($\lambda \approx 0.000019, \mu_J \approx 0.0068$), causing the model to collapse back to the base diffusion model. This indicates that the extension does not overfit the training period, but it also does not improve daily cross-sectional yield predictions. In practice, jump-diffusion is valuable for option pricing and tail-risk hedging rather than daily curve reconstruction.

---

### 8.3 Extensions and Modelling Choices

#### 8.3.1 Mathematical Justification of Jump-Diffusion
A jump-diffusion process is justified because interest rates are subject to sudden, discontinuous shocks from macroeconomic announcements, Fed meetings, or market crises. Pure diffusion models can only represent continuous paths and fail to capture these jumps, leading to underpriced options and underestimation of tail risk. The affine jump-diffusion framework of Duffie-Pan-Singleton maintains the analytical tractability of bond pricing while adding jump-risk flexibility.

#### 8.3.2 Qualitative Impact of Jumps on Yield Curve
During stress periods, the expectations of jumps introduce a risk premium that shifts the short end of the yield curve upward or downward depending on the direction of expected shocks. For positive jump distributions, the expectation of potential rate hikes flattens the curve or steepens the long-end to compensate investors for jump-risk, allowing the curve to capture shapes that pure-diffusion models cannot generate.

#### 8.3.3 Estimation Challenges of Advanced Models
- **Two-Factor Models**: Introduce latent variables, requiring state-space formulations (such as Kalman Filtering) that are highly sensitive to transition noise and covariance assumptions, increasing optimization instability.
- **Time-Dependent Models (CIR++)**: Allow parameters to vary continuously over time to fit the initial term structure perfectly. This often leads to overfitting, making the model unstable for forecasting and causing future yield projections to violate no-arbitrage bounds.
